In [1]:
!pip install trackio -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.0/875.0 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


## HuggingFace Token Setup

In [2]:
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_token")

## Main Pipeline Script (Baseline+Pretrained) 

In [3]:
%%writefile main.py
# --------------------------------------------------------------------
# ------------------------ Important Imports -------------------------
# --------------------------------------------------------------------
import os, random, logging,trackio
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd
import cv2, torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, mean_squared_error
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor, Callback
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.strategies import DDPStrategy
import timm

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

# --------------------------------------------------------------------
# --------------------- Pipeline Configurations ----------------------
#---------------------------------------------------------------------
class Config:
    DATA_DIR = Path('/kaggle/input/sep-25-dl-gen-ai-nppe-1/face_dataset')
    OUTPUT_DIR = Path('/kaggle/working')
    MODEL_NAME = 'convnextv2_base.fcmae_ft_in22k_in1k' 
    MODEL_TYPE = 'optimized'
    IMG_SIZE = 224
    NUM_AGE_CLASSES = 100
    PRETRAINED = True
    DROPOUT = 0.40
    BATCH_SIZE = 64
    ACCUMULATE_GRAD_BATCHES = 2
    NUM_WORKERS = 4
    N_FOLDS = 5
    FOLD = 0
    EPOCHS = 40
    LR = 1.8e-4
    MIN_LR = 1e-7
    WEIGHT_DECAY = 0.015
    GRADIENT_CLIP_VAL = 2.0
    AGE_WEIGHT = 1.8
    GENDER_WEIGHT = 1.2
    BIN_WEIGHT = 0.08
    USE_TTA = True
    USE_GEM_POOLING = True
    SEED = 42
    PATIENCE = 4
    PRECISION = '16-mixed'
    WARMUP_EPOCHS = 3
    MIXUP_ALPHA = 0.25
    LABEL_SMOOTHING = 0.12
    USE_GRADIENT_CHECKPOINTING = True
    
    # Baseline model Configs
    BASELINE_DROPOUT = 0.30
    BASELINE_BATCH_SIZE = 128
    BASELINE_LR = 3e-4
    BASELINE_WEIGHT_DECAY = 1e-4
    BASELINE_GRADIENT_CLIP_VAL = 1.0
    BASELINE_AGE_WEIGHT = 1.0
    BASELINE_GENDER_WEIGHT = 1.5
    BASELINE_USE_TTA = False

    # TrackIO Configs
    TRACKIO_PROJECT = "25-t3-nppe1"
    TRACKIO_SPACE_ID = "tushar-sharma/dlgenai-nppe"
    TRACKIO_NAME = "Baseline-CNN"
    TRACKIO_GROUP = "baseline"
    
    @classmethod
    def from_args(cls, args):
        # Main model settings
        cls.BATCH_SIZE = args.batch_size
        cls.EPOCHS = args.epochs
        cls.FOLD = args.fold
        cls.LR = args.lr
        cls.USE_TTA = getattr(args, 'use_tta', True)
        cls.MODEL_TYPE = getattr(args, 'model', 'optimized')
        cls.TRACKIO_NAME = getattr(args, 'trackio_name', 'Baseline-CNN')
        cls.TRACKIO_GROUP = getattr(args, 'trackio_group', 'baseline')
        
        # Overridden HP for the Baseline Model
        if cls.MODEL_TYPE == 'baseline':
            cls.DROPOUT = cls.BASELINE_DROPOUT
            cls.LR = cls.BASELINE_LR
            cls.WEIGHT_DECAY = cls.BASELINE_WEIGHT_DECAY
            cls.GRADIENT_CLIP_VAL = cls.BASELINE_GRADIENT_CLIP_VAL
            cls.AGE_WEIGHT = cls.BASELINE_AGE_WEIGHT
            cls.GENDER_WEIGHT = cls.BASELINE_GENDER_WEIGHT
            cls.USE_TTA = cls.BASELINE_USE_TTA
        return cls

# --------------------------------------------------------------------
# ------------------ Augumentation & Preprocessing -------------------
# --------------------------------------------------------------------
def get_transforms(img_size=224, is_train=True):
    if is_train:
        return A.Compose([
            A.RandomResizedCrop((img_size, img_size), scale=(0.6, 1.0), ratio=(0.9, 1.1), p=1.0),
            A.HorizontalFlip(p=0.5),
            A.Affine(rotate=(-25, 25), translate_percent=(-0.08, 0.08), scale=(0.8, 1.2), shear=(-10, 10), p=0.7),
            A.OneOf([
                A.RandomBrightnessContrast(0.35, 0.35, p=1.0),
                A.HueSaturationValue(25, 35, 30, p=1.0),
                A.RGBShift(30, 30, 30, p=1.0),
            ], p=0.85),
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 9), p=1.0),
                A.MotionBlur(blur_limit=(3, 9), p=1.0),
                A.GaussNoise(std_range=(0.06, 0.24), p=1.0),
            ], p=0.45),
            A.CoarseDropout(
                num_holes_range=(3, 6),
                hole_height_range=(int(img_size*0.05), int(img_size*0.12)),
                hole_width_range=(int(img_size*0.05), int(img_size*0.12)),
                fill=0, p=0.5
            ),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), 
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])

def mixup_data(x, y_age, y_gender, y_bin, alpha=0.25):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    return mixed_x, y_age, y_age[index], y_gender, y_gender[index], y_bin, y_bin[index], lam

# --------------------------------------------------------------------
# --------------------------- Data Modules ---------------------------
# --------------------------------------------------------------------
class FaceDataset(Dataset):
    def __init__(self, df, img_dir, transforms, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transforms = transforms
        self.is_test = is_test
    
    def __len__(self): 
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.is_test:
            img_id = row['id']
            test_dir = self.img_dir / 'test'
            possible_paths = [
                test_dir / f"{img_id}.jpg",
                test_dir / f"{str(img_id).zfill(5)}.jpg",
            ]
            img = None
            for path in possible_paths:
                if path.exists():
                    img = cv2.imread(str(path))
                    break
            if img is None:
                img = np.zeros((224, 224, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img_path = self.img_dir / row['full_path']
            img = cv2.imread(str(img_path))
            if img is None:
                img = np.zeros((224, 224, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(image=img)['image']
        if self.is_test: 
            return img, row['id']
        age = torch.tensor(row['age'], dtype=torch.float32)
        gender = torch.tensor(row['gender'], dtype=torch.long)
        age_bin = torch.tensor(min(max(row['age']//10, 0), 9), dtype=torch.long)
        return img, age, gender, age_bin

class FaceDataModule(pl.LightningDataModule):
    def __init__(self, config): 
        super().__init__()
        self.config = config
    
    def setup(self, stage=None):
        df = pd.read_csv(self.config.DATA_DIR / 'train.csv')
        df['age_group'] = pd.cut(df['age'], bins=[0,10,20,30,40,50,100], labels=False)
        df['stratify_col'] = df['gender'].astype(str)+'_'+df['age_group'].astype(str)
        df['fold'] = -1
        skf = StratifiedKFold(n_splits=self.config.N_FOLDS, shuffle=True, random_state=self.config.SEED)
        for fold_idx,(_,val_idx) in enumerate(skf.split(df, df['stratify_col'])): 
            df.loc[val_idx,'fold']=fold_idx
        self.train_df = df[df['fold']!=self.config.FOLD].reset_index(drop=True)
        self.val_df = df[df['fold']==self.config.FOLD].reset_index(drop=True)
        self.test_df = pd.read_csv(self.config.DATA_DIR / 'test.csv')
        logger.info(f"Data | Train: {len(self.train_df)} | Val: {len(self.val_df)} | Test: {len(self.test_df)}")
    
    def train_dataloader(self): 
        return DataLoader(
            FaceDataset(self.train_df, self.config.DATA_DIR, get_transforms(self.config.IMG_SIZE, True)),
            batch_size=self.config.BATCH_SIZE, shuffle=True, num_workers=self.config.NUM_WORKERS,
            pin_memory=True, persistent_workers=True
        )
    
    def val_dataloader(self):   
        return DataLoader(
            FaceDataset(self.val_df, self.config.DATA_DIR, get_transforms(self.config.IMG_SIZE, False)),
            batch_size=self.config.BATCH_SIZE, shuffle=False, num_workers=self.config.NUM_WORKERS,
            pin_memory=True, persistent_workers=True
        )
    
    def test_dataloader(self):  
        return DataLoader(
            FaceDataset(self.test_df, self.config.DATA_DIR, get_transforms(self.config.IMG_SIZE, False), is_test=True),
            batch_size=self.config.BATCH_SIZE, shuffle=False, num_workers=self.config.NUM_WORKERS, pin_memory=True
        )

# --------------------------------------------------------------------
# -------------------------- Baseline Model --------------------------
# --------------------------------------------------------------------
# ------------------------------------ Baseline CNN Model Architecture
# --------------------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, pool=True):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2, 2) if pool else nn.Identity()
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.pool(x)
        return x

class SimpleBaselineCNN(nn.Module):
    def __init__(self, num_age_classes=100, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32, 3, pool=True),
            ConvBlock(32, 64, 3, pool=True),
            ConvBlock(64, 128, 3, pool=True),
            ConvBlock(128, 128, 3, pool=False),
            ConvBlock(128, 256, 3, pool=True),
            ConvBlock(256, 256, 3, pool=False),
            ConvBlock(256, 512, 3, pool=True),
            ConvBlock(512, 512, 3, pool=False),
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.feature_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5)
        )
        self.age_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.7),
            nn.Linear(128, num_age_classes - 1)
        )
        self.gender_head = nn.Sequential(
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.8),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2)
        )
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
                    
    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = x.flatten(1)
        features = self.feature_fc(x)
        age_logits = self.age_head(features)
        gender_logits = self.gender_head(features)
        return age_logits, gender_logits

# --------------------------------------------------------------------
# -------------------------- PT Lightning Module of Baseline CNN Model
# --------------------------------------------------------------------
class BaselineFaceModule(pl.LightningModule):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.model = SimpleBaselineCNN(
            num_age_classes=config.NUM_AGE_CLASSES,
            dropout=config.DROPOUT
        )
        self.validation_step_outputs = []

    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        images, age_true, gender_true, age_bin_true = batch
        age_logits, gender_logits = self(images)
        age_loss = coral_bce_loss(age_logits, age_true)
        gender_loss = F.cross_entropy(gender_logits, gender_true)
        loss = self.config.AGE_WEIGHT * age_loss + self.config.GENDER_WEIGHT * gender_loss
        self.log('train_loss', loss, prog_bar=True, sync_dist=True)
        trackio.log({"train_loss": float(loss.item())})
        return loss
    
    def validation_step(self, batch, batch_idx):
        images, age_true, gender_true, age_bin_true = batch
        age_logits, gender_logits = self(images)
        age_pred = coral_probs_to_label(age_logits)
        gender_pred = torch.argmax(gender_logits, dim=1)
        loss = (
            self.config.AGE_WEIGHT * coral_bce_loss(age_logits, age_true) +
            self.config.GENDER_WEIGHT * F.cross_entropy(gender_logits, gender_true)
        )
        trackio.log({"val_loss": float(loss.item())})
        self.validation_step_outputs.append({
            'loss': loss.detach(),
            'age_pred': age_pred.cpu().numpy(),
            'age_true': age_true.cpu().numpy(),
            'gender_pred': gender_pred.cpu().numpy(),
            'gender_true': gender_true.cpu().numpy()
        })
        return loss

    def on_validation_epoch_end(self):
        if len(self.validation_step_outputs) == 0:
            return
        age_preds = np.concatenate([x['age_pred'] for x in self.validation_step_outputs])
        age_trues = np.concatenate([x['age_true'] for x in self.validation_step_outputs])
        gender_preds = np.concatenate([x['gender_pred'] for x in self.validation_step_outputs])
        gender_trues = np.concatenate([x['gender_true'] for x in self.validation_step_outputs])
        age_preds = np.clip(age_preds, 1, self.config.NUM_AGE_CLASSES)
        metrics = calculate_metrics(age_trues, age_preds, gender_trues, gender_preds)
        avg_loss = torch.stack([x['loss'] for x in self.validation_step_outputs]).mean()
        self.log('val_loss', avg_loss, prog_bar=True, sync_dist=True)
        self.log('val_age_rmse', metrics['age_rmse'], prog_bar=True, sync_dist=True)
        self.log('val_age_score', metrics['age_score'], prog_bar=True, sync_dist=True)
        self.log('val_gender_f1', metrics['gender_f1'], prog_bar=True, sync_dist=True)
        self.log('val_final_score', metrics['final_score'], prog_bar=True, sync_dist=True)
        trackio.log({
            "val_age_rmse": float(metrics['age_rmse']),
            "val_age_score": float(metrics['age_score']),
            "val_gender_f1": float(metrics['gender_f1']),
            "val_final_score": float(metrics['final_score']),
        })
        self.validation_step_outputs.clear()
    
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.config.LR,
            weight_decay=self.config.WEIGHT_DECAY,
            betas=(0.9, 0.999)
        )
        warmup_steps = self.config.WARMUP_EPOCHS
        total_steps = self.config.EPOCHS
        def lr_lambda(epoch):
            if epoch < warmup_steps:
                return (epoch + 1) / warmup_steps
            else:
                progress = (epoch - warmup_steps) / (total_steps - warmup_steps)
                return 0.5 * (1 + np.cos(np.pi * progress))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'}}

# --------------------------------------------------------------------
# ---------------------- Pre-trained SOTA Model ----------------------
# --------------------------------------------------------------------
# -------------------------------- Pre-trained SOTA Model Architecture
# --------------------------------------------------------------------
class AgeGenderModel(nn.Module):
    def __init__(self, backbone_name, pretrained=True, dropout=0.40, use_gem=True, num_classes=100, use_checkpointing=True):
        super().__init__()
        if timm is None:
            raise ImportError("TIMM library not found. Install with 'pip install timm'.")
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained, num_classes=0, global_pool='',
            drop_rate=dropout*0.25, drop_path_rate=0.2
        )
        if use_checkpointing and hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(True)
            logger.info("✓ Gradient checkpointing enabled")
        self.num_features = self.backbone.num_features
        self.pool = GeM(p=3.5) if use_gem else nn.AdaptiveAvgPool2d(1)
        self.feature_bn = nn.BatchNorm1d(self.num_features)
        self.feature_dropout = nn.Dropout(dropout*0.4)
        self.age_head = CoralOrdinalHead(self.num_features, num_classes=num_classes, dropout=dropout)
        self.gender_head = nn.Sequential(
            nn.Linear(self.num_features, 192), nn.BatchNorm1d(192), nn.SiLU(), nn.Dropout(dropout*0.8),
            nn.Linear(192, 64), nn.BatchNorm1d(64), nn.SiLU(), nn.Dropout(dropout*0.5),
            nn.Linear(64, 2)
        )
        self.bin_head = nn.Sequential(
            nn.Linear(self.num_features, 96), nn.BatchNorm1d(96), nn.SiLU(), 
            nn.Dropout(dropout*0.4), nn.Linear(96, 10)
        )
    def forward(self, x):
        features = self.backbone.forward_features(x)
        features = self.pool(features).flatten(1) if len(features.shape)==4 else features.mean(dim=1)
        features = self.feature_bn(features)
        features = self.feature_dropout(features)
        return self.age_head(features), self.gender_head(features), self.bin_head(features)

# --------------------------------------------------------------------
# ---------------------- PT Lightning Module of Pre-trained SOTA Model
# --------------------------------------------------------------------
class FaceModule(pl.LightningModule):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.model = AgeGenderModel(
            config.MODEL_NAME, config.PRETRAINED, config.DROPOUT, 
            config.USE_GEM_POOLING, config.NUM_AGE_CLASSES, config.USE_GRADIENT_CHECKPOINTING
        )
        self.validation_step_outputs = []
        self.use_mixup = config.MIXUP_ALPHA > 0

    def forward(self, x): 
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        images, age_true, gender_true, age_bin_true = batch
        if self.use_mixup and self.training and np.random.rand() < 0.5:
            images, age_a, age_b, gender_a, gender_b, bin_a, bin_b, lam = mixup_data(
                images, age_true, gender_true, age_bin_true, self.config.MIXUP_ALPHA
            )
            age_logits, gender_logits, bin_logits = self(images)
            age_loss = lam * coral_bce_loss(age_logits, age_a) + (1 - lam) * coral_bce_loss(age_logits, age_b)
            gender_loss = lam * F.cross_entropy(gender_logits, gender_a, label_smoothing=self.config.LABEL_SMOOTHING)
            gender_loss += (1 - lam) * F.cross_entropy(gender_logits, gender_b, label_smoothing=self.config.LABEL_SMOOTHING)
            bin_loss = lam * F.cross_entropy(bin_logits, bin_a) + (1 - lam) * F.cross_entropy(bin_logits, bin_b)
        else:
            age_logits, gender_logits, bin_logits = self(images)
            age_loss = coral_bce_loss(age_logits, age_true)
            gender_loss = F.cross_entropy(gender_logits, gender_true, label_smoothing=self.config.LABEL_SMOOTHING)
            bin_loss = F.cross_entropy(bin_logits, age_bin_true)
        loss = self.config.AGE_WEIGHT * age_loss + self.config.GENDER_WEIGHT * gender_loss + self.config.BIN_WEIGHT * bin_loss
        self.log('train_loss', loss, prog_bar=True, sync_dist=True)
        trackio.log({"train_loss": float(loss.item())})
        return loss
    
    def validation_step(self, batch, batch_idx):
        images, age_true, gender_true, age_bin_true = batch
        age_logits, gender_logits, bin_logits = self(images)
        age_pred = coral_probs_to_label(age_logits)
        gender_pred = torch.argmax(gender_logits, dim=1)
        loss = (
            self.config.AGE_WEIGHT * coral_bce_loss(age_logits, age_true) +
            self.config.GENDER_WEIGHT * F.cross_entropy(gender_logits, gender_true) +
            self.config.BIN_WEIGHT * F.cross_entropy(bin_logits, age_bin_true)
        )
        trackio.log({"val_loss": float(loss.item())})
        self.validation_step_outputs.append({
            'loss': loss.detach(),
            'age_pred': age_pred.cpu().numpy(),
            'age_true': age_true.cpu().numpy(),
            'gender_pred': gender_pred.cpu().numpy(),
            'gender_true': gender_true.cpu().numpy()
        })
        return loss
    
    def on_validation_epoch_end(self):
        if len(self.validation_step_outputs) == 0: 
            return
        age_preds = np.concatenate([x['age_pred'] for x in self.validation_step_outputs])
        age_trues = np.concatenate([x['age_true'] for x in self.validation_step_outputs])
        gender_preds = np.concatenate([x['gender_pred'] for x in self.validation_step_outputs])
        gender_trues = np.concatenate([x['gender_true'] for x in self.validation_step_outputs])
        age_preds = np.clip(age_preds, 1, self.config.NUM_AGE_CLASSES)
        metrics = calculate_metrics(age_trues, age_preds, gender_trues, gender_preds)
        avg_loss = torch.stack([x['loss'] for x in self.validation_step_outputs]).mean()
        self.log('val_loss', avg_loss, prog_bar=True, sync_dist=True)
        self.log('val_age_rmse', metrics['age_rmse'], prog_bar=True, sync_dist=True)
        self.log('val_age_score', metrics['age_score'], prog_bar=True, sync_dist=True)
        self.log('val_gender_f1', metrics['gender_f1'], prog_bar=True, sync_dist=True)
        self.log('val_final_score', metrics['final_score'], prog_bar=True, sync_dist=True)
        trackio.log({
            "val_age_rmse": float(metrics['age_rmse']),
            "val_age_score": float(metrics['age_score']),
            "val_gender_f1": float(metrics['gender_f1']),
            "val_final_score": float(metrics['final_score']),
        })
        self.validation_step_outputs.clear()
    
    def configure_optimizers(self):
        no_decay = ['bias', 'bn', 'norm']
        optimizer_params = [
            {'params': [p for n, p in self.model.named_parameters() if not any(nd in n for nd in no_decay) and 'backbone' in n],
             'lr': self.config.LR * 0.5, 'weight_decay': self.config.WEIGHT_DECAY},
            {'params': [p for n, p in self.model.named_parameters() if any(nd in n for nd in no_decay) and 'backbone' in n],
             'lr': self.config.LR * 0.5, 'weight_decay': 0.0},
            {'params': [p for n, p in self.model.named_parameters() if 'backbone' not in n],
             'lr': self.config.LR, 'weight_decay': self.config.WEIGHT_DECAY}
        ]
        opt = torch.optim.AdamW(optimizer_params, betas=(0.9, 0.999), eps=1e-8)
        warmup_steps = self.config.WARMUP_EPOCHS
        total_steps = self.config.EPOCHS
        def lr_lambda(epoch):
            if epoch < warmup_steps:
                return (epoch + 1) / warmup_steps
            else:
                progress = (epoch - warmup_steps) / (total_steps - warmup_steps)
                return 0.5 * (1 + np.cos(np.pi * progress))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sched, 'interval': 'epoch'}}

# --------------------------------------------------------------------
# ----------------------------- Custom HEADS of Pre-trained SOTA Model
# --------------------------------------------------------------------
class GeM(nn.Module):
    def __init__(self, p=3.5, eps=1e-6): 
        super().__init__()
        self.p = nn.Parameter(torch.ones(1)*p)
        self.eps = eps
    def forward(self, x): 
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2),x.size(-1))).pow(1.0/self.p)

class CoralOrdinalHead(nn.Module):
    def __init__(self, in_features, num_classes=100, dropout=0.40):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_features, 512), 
            nn.BatchNorm1d(512), 
            nn.SiLU(), 
            nn.Dropout(dropout*0.8),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(dropout*0.6),
            nn.Linear(256, num_classes-1)
        )
    def forward(self, x): 
        return self.fc(x)

# --------------------------------------------------------------------
# -------------------- Custom Loss Functions of Pre-trained SOTA Model
# --------------------------------------------------------------------
def coral_bce_loss(logits, targets):
    cum_labels = torch.zeros_like(logits)
    for b in range(logits.size(0)):
        idx = int(min(max(int(targets[b]), 0), logits.size(1)))
        if idx > 0:
            cum_labels[b,:idx] = 1
    return F.binary_cross_entropy_with_logits(logits, cum_labels.float())

def coral_probs_to_label(logits):
    prob = torch.sigmoid(logits)
    return torch.sum(prob > 0.5, dim=1)

# --------------------------------------------------------------------
# ---------------------------- Utilities -----------------------------
# --------------------------------------------------------------------
# -------------------------------- Harmonic Mean of F1 Score and nRMSE
# --------------------------------------------------------------------
def calculate_metrics(age_true, age_pred, gender_true, gender_pred):
    age_rmse = np.sqrt(mean_squared_error(age_true, age_pred))
    age_score = max(0.0, 1.0 - min(age_rmse, 30.0)/30.0)
    gender_f1 = f1_score(gender_true, gender_pred, average='macro', zero_division=0)
    final_score = 2*age_score*gender_f1/(age_score+gender_f1) if age_score+gender_f1>1e-6 else 0.0
    return {'age_rmse': age_rmse, 'age_score': age_score, 'gender_f1': gender_f1, 'final_score': final_score}

# --------------------------------------------------------------------
# -------------------------------------- Callback for Tracking Metrics
# --------------------------------------------------------------------
class MetricsCallback(Callback):
    def __init__(self):
        super().__init__()
        self.best_score = 0.0
        self.epoch_start = None
    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start = datetime.now()
        logger.info("="*80)
        logger.info(f"Epoch {trainer.current_epoch + 1}/{trainer.max_epochs}")
    def on_train_epoch_end(self, trainer, pl_module):
        elapsed = (datetime.now() - self.epoch_start).total_seconds()
        metrics = trainer.callback_metrics
        train_loss = metrics.get('train_loss', 0)
        lr = trainer.optimizers[0].param_groups[0]['lr']
        logger.info(f"TRAIN | Loss: {train_loss:.4f} | LR: {lr:.2e} | Time: {elapsed:.1f}s")
    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        val_loss = metrics.get('val_loss', 0)
        val_rmse = metrics.get('val_age_rmse', 0)
        val_age_score = metrics.get('val_age_score', 0)
        val_gender_f1 = metrics.get('val_gender_f1', 0)
        val_final = metrics.get('val_final_score', 0)
        logger.info(f"VAL   | Loss: {val_loss:.4f} | RMSE: {val_rmse:.4f} | Age: {val_age_score:.4f} | Gender F1: {val_gender_f1:.4f} | Score: {val_final:.4f}")
        if val_final > self.best_score:
            improvement = val_final - self.best_score
            self.best_score = val_final
            logger.info(f"      | ⭐ NEW BEST: {self.best_score:.4f} (+{improvement:.4f})")
        logger.info("="*80)

# --------------------------------------------------------------------
# ----------------------------------- Global Reproducible Seed Setting
# --------------------------------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# --------------------------------------------------------------------
# ------------------------------------------------ TTA Model Inference
# --------------------------------------------------------------------
def generate_predictions_with_tta(model, dataloader, device='cuda', config=None):
    model.eval()
    model = model.to(device)
    predictions = {'id': [], 'age': [], 'gender': []}
    logger.info(f"Running inference with TTA={config.USE_TTA}")
    with torch.no_grad():
        for batch_idx, (images, ids) in enumerate(dataloader):
            batch_age_preds = []
            batch_gender_preds = []
            images_gpu = images.to(device)
            age_logits, gender_logits, _ = model(images_gpu)
            batch_age_preds.append(coral_probs_to_label(age_logits).cpu().numpy())
            batch_gender_preds.append(torch.softmax(gender_logits, dim=1).cpu().numpy())
            if config.USE_TTA:
                images_flip = torch.flip(images, dims=[3]).to(device)
                age_logits, gender_logits, _ = model(images_flip)
                batch_age_preds.append(coral_probs_to_label(age_logits).cpu().numpy())
                batch_gender_preds.append(torch.softmax(gender_logits, dim=1).cpu().numpy())
            age_pred = np.mean(batch_age_preds, axis=0)
            gender_pred = np.argmax(np.mean(batch_gender_preds, axis=0), axis=1)
            predictions['id'].extend(ids.cpu().numpy())
            predictions['age'].extend(age_pred)
            predictions['gender'].extend(gender_pred)
    df = pd.DataFrame(predictions)
    df['age'] = np.clip(np.round(df['age']), 1, config.NUM_AGE_CLASSES).astype(int)
    logger.info(f"Predictions | Age mean: {df['age'].mean():.1f} ± {df['age'].std():.2f}")
    logger.info(f"Gender | M: {sum(df['gender']==1)} | F: {sum(df['gender']==0)}")
    return df

# --------------------------------------------------------------------
# ----------------------------------- Submission File & Test Inference
# --------------------------------------------------------------------
def generate_predictions(model, dataloader, device='cuda', config=None):
    model.eval()
    model = model.to(device)
    predictions = {'id': [], 'age': [], 'gender': []}
    logger.info("Running baseline inference...")
    with torch.no_grad():
        for images, ids in dataloader:
            images = images.to(device)
            age_logits, gender_logits = model(images)
            age_pred = coral_probs_to_label(age_logits).cpu().numpy()
            gender_pred = torch.argmax(gender_logits, dim=1).cpu().numpy()
            predictions['id'].extend(ids.cpu().numpy())
            predictions['age'].extend(age_pred)
            predictions['gender'].extend(gender_pred)
    df = pd.DataFrame(predictions)
    df['age'] = np.clip(np.round(df['age']), 1, config.NUM_AGE_CLASSES).astype(int)
    logger.info(f"Predictions | Age mean: {df['age'].mean():.1f} ± {df['age'].std():.2f}")
    logger.info(f"Gender | M: {sum(df['gender']==1)} | F: {sum(df['gender']==0)}")
    return df

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# --------------------------------------------------------------------
# -------------- Main Entry Point (Training+Inference) ---------------
# --------------------------------------------------------------------
def train():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--batch-size', type=int, default=64)
    parser.add_argument('--epochs', type=int, default=40)
    parser.add_argument('--fold', type=int, default=0)
    parser.add_argument('--lr', type=float, default=1.8e-4)
    parser.add_argument('--use-tta', action='store_true', default=True)
    parser.add_argument('--model', type=str, choices=['optimized','baseline'], default='baseline')
    parser.add_argument('--trackio_name', type=str, default="Pretrained-SOTA-Model")
    parser.add_argument('--trackio_group', type=str, default="optimized")
    
    args = parser.parse_args()
    config = Config.from_args(args)
    set_seed(config.SEED)
    torch.cuda.empty_cache()
    
    logger.info("="*80)
    trackio.init(
        project=config.TRACKIO_PROJECT,
        space_id=config.TRACKIO_SPACE_ID,
        name=config.TRACKIO_NAME,
        group=config.TRACKIO_GROUP,
    )
    logger.info("TrackIO Initialization Sucessfull!")
    logger.info("="*80)
    mode_msg = "ConvNeXtV2 Base Model" if config.MODEL_TYPE == 'optimized' else "Baseline Simple CNN Model"
    logger.info(f"Age & Gender Regression Pipeline Mode - {mode_msg}")
    logger.info("="*80)
    if config.MODEL_TYPE == 'optimized':
        logger.info(f"Model Architecture: {config.MODEL_NAME}")
        logger.info(f"Batch Size: {config.BATCH_SIZE} | Accum Steps: {config.ACCUMULATE_GRAD_BATCHES}")
        logger.info(f"Gradient Checkpointing: {config.USE_GRADIENT_CHECKPOINTING}")
        logger.info(f"Mixed Precision: {config.PRECISION}")
    else:
        n_params = count_parameters(SimpleBaselineCNN(num_age_classes=config.NUM_AGE_CLASSES, dropout=config.DROPOUT))
        logger.info(f"Model Architecture: Simple 5-block CNN")
        logger.info(f"Total Parameters: {n_params:,} ({n_params/1e6:.2f}M)")
        logger.info(f"Batch Size: {config.BATCH_SIZE} | Mixed Precision: {config.PRECISION}")
    logger.info("="*80)
    
    data_module = FaceDataModule(config)
    model = FaceModule(config) if config.MODEL_TYPE == 'optimized' else BaselineFaceModule(config)
    checkpoint_callback = ModelCheckpoint(
        dirpath=config.OUTPUT_DIR / 'checkpoints',
        filename=f'{config.MODEL_TYPE}_fold{config.FOLD}_' + 'e{epoch:02d}_s{val_final_score:.4f}',
        monitor='val_final_score', mode='max', save_top_k=3, save_last=True
    )
    early_stop = EarlyStopping(
        monitor='val_final_score', patience=config.PATIENCE, mode='max', min_delta=0.0002
    )
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    metrics_callback = MetricsCallback()
    csv_logger = CSVLogger(save_dir=config.OUTPUT_DIR / 'logs', name='training')
    
    trainer = pl.Trainer(
        max_epochs=config.EPOCHS,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=2,
        strategy=DDPStrategy(find_unused_parameters=config.MODEL_TYPE == 'optimized'),
        precision=config.PRECISION,
        callbacks=[checkpoint_callback, early_stop, lr_monitor, metrics_callback],
        logger=csv_logger,
        gradient_clip_val=config.GRADIENT_CLIP_VAL,
        gradient_clip_algorithm='norm',
        accumulate_grad_batches=config.ACCUMULATE_GRAD_BATCHES,
        log_every_n_steps=25,
        deterministic=False,
        benchmark=True,
        enable_progress_bar=True,
        enable_model_summary=False
    )
    
    logger.info("Starting training...")
    trainer.fit(model, data_module)
    logger.info("Training complete!")
    
    logger.info(f"Loading best: {checkpoint_callback.best_model_path}")
    best_model = (FaceModule.load_from_checkpoint if config.MODEL_TYPE=='optimized' else BaselineFaceModule.load_from_checkpoint)(
        checkpoint_callback.best_model_path, config=config
    )
    preds_fn = generate_predictions_with_tta if config.MODEL_TYPE == 'optimized' else generate_predictions
    predictions = preds_fn(best_model.model, data_module.test_dataloader(), config=config)
    output_path = config.OUTPUT_DIR / f'{config.MODEL_TYPE}_submission.csv'
    predictions.to_csv(output_path, index=False)
    logger.info("="*80)
    logger.info(f"✅ Saved: {output_path}")
    logger.info(f"🏆 Best Score: {checkpoint_callback.best_model_score:.4f}")
    logger.info("="*80)
    trackio.finish()

if __name__ == '__main__':
    train()

Writing main.py


## Baseline CNN Model Execution

In [4]:
# !python main.py --batch-size 64 --epochs 50 --model "baseline" --trackio_name "Baseline-CNN-Model" --trackio_group "baseline"

## Pretrained SOTA Model Execution

In [5]:
# !python main.py --batch-size 64 --epochs 50 --model "optimized" --trackio_name "Pretrained-SOTA-Model" --trackio_group "optimized"

## Model Upload
We'll take the output files of the Version-2 of the same notebook that were generated while training and upload the best trained model on KaggleHub (Baseline and Optimized).

In [6]:
# import kagglehub
# best_baseline_model_path = "/kaggle/input/nppe-nb2/fold_1/checkpoints/optimized_f1_eepoch=10_sval_final_score=0.8539.ckpt"
# best_optimized_model_path = "/kaggle/input/nppe-nb2/fold_2/checkpoints/optimized_f2_eepoch=23_sval_final_score=0.8548.ckpt"

# KAGGLE_USERNAME = 'tushar5harma'
# MODEL = 'Age-Gender-Regression-Models'
# FRAMEWORK = 'pytorch'

# BASELINE_VARIATION = 'Baseline-Model'
# OPTIMIZED_VARIATION = 'Optimized-Model'

In [7]:
# handle = f'{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{BASELINE_VARIATION}'
# kagglehub.model_upload(handle, best_baseline_model_path, version_notes='5-Block CNN')

In [8]:
# handle = f'{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{OPTIMIZED_VARIATION}'
# kagglehub.model_upload(handle, best_optimized_model_path, version_notes='ConvNeXT_V2+CoralOrdinal+GeM TTS')

## Model Inference (Optimized Model)

In [9]:
# from main import *

# MODEL_PATH = '/kaggle/input/age-gender-regression-models/pytorch/optimized-model/1/optimized_fold0_eepoch14_sval_final_score0.8549.ckpt'
# OUTPUT_FILE = 'submission.csv'

# logger.info("="*80)
# logger.info("Loading Optimized Model Checkpoints")
# logger.info("="*80)

# config = Config()
# config.PRETRAINED = False

# # Load model from checkpoint
# model = FaceModule.load_from_checkpoint(
#     MODEL_PATH,
#     config=config,
#     map_location='cuda' if torch.cuda.is_available() else 'cpu'
# )

# model.eval()
# model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
# logger.info("✓ Model loaded successfully")

# logger.info("\n\n"+"="*98)
# logger.info("Loading Test data")
# logger.info("="*80)

# test_df = pd.read_csv(config.DATA_DIR / 'test.csv')
# logger.info(f"Test samples: {len(test_df)}")

# test_dataset = FaceDataset(
#     test_df,
#     config.DATA_DIR,
#     get_transforms(config.IMG_SIZE, is_train=False),
#     is_test=True
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=config.BATCH_SIZE,
#     shuffle=False,
#     num_workers=config.NUM_WORKERS,
#     pin_memory=True
# )

# logger.info("✓ Test data Loaded Sucessfully")

# logger.info("\n\n"+"="*98)
# logger.info("Initiating Inference Pipline")
# logger.info("="*80)
# predictions = generate_predictions_with_tta(
#     model.model,
#     test_loader,
#     device='cuda' if torch.cuda.is_available() else 'cpu',
#     config=config
# )

# logger.info("✓ Predictions complete")

# logger.info("\n\n"+"="*98)
# logger.info("Generating Submission File")
# logger.info("="*80)
# predictions.to_csv(OUTPUT_FILE, index=False)
# logger.info(f"✓ Submission saved: {OUTPUT_FILE}")
# logger.info("\nPrediction statistics:")
# logger.info(f"Age - Mean: {predictions['age'].mean():.1f}, Std: {predictions['age'].std():.2f}")
# logger.info(f"Age - Min: {predictions['age'].min()}, Max: {predictions['age'].max()}")
# logger.info(f"Gender - Male (1): {sum(predictions['gender']==1)}, Female (0): {sum(predictions['gender']==0)}")

## Training Logs and Metrics 
The HF API for TrackIO is not functional and due to technical difficulties the generated logs that were generated during training in the Version 2 of this notbook using the PT Lighting CSVLogger Callback Module are being used to visualize the training logs of the given metrics of the competition.

In [10]:
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt
# import numpy as np

# df_base = pd.read_csv('/kaggle/input/training-metrics-logs/baseline_metrics.csv', sep=None, engine='python')
# df_opt = pd.read_csv('/kaggle/input/training-metrics-logs/optimized_metrics.csv', sep=None, engine='python')

# df_base.replace([np.inf, -np.inf], np.nan, inplace=True)
# df_opt.replace([np.inf, -np.inf], np.nan, inplace=True)

# core_cols = ['epoch', 'step', 'lr', 'lr-AdamW']
# metrics = [col for col in df_base.columns if col not in core_cols and df_base[col].dtype != object]

# sns.set_theme(context='notebook', style='whitegrid')
# fig, axs = plt.subplots(len(metrics), 2, figsize=(12, 4 * len(metrics)), sharex='col')
# if len(metrics) == 1:
#     axs = np.reshape(axs, (1, 2))

# for i, metric in enumerate(metrics):
#     # Baseline
#     sns.lineplot(x=df_base['epoch'], y=df_base[metric], ax=axs[i,0], color='skyblue', label='Baseline')
#     axs[i,0].set_title(f'Baseline: {metric}')
#     axs[i,0].set_ylabel(metric)
#     axs[i,0].set_xlabel('Epoch')
#     axs[i,0].legend()
#     # Optimized
#     sns.lineplot(x=df_opt['epoch'], y=df_opt[metric], ax=axs[i,1], color='orange', label='Optimized')
#     axs[i,1].set_title(f'Optimized: {metric}')
#     axs[i,1].set_ylabel(metric)
#     axs[i,1].set_xlabel('Epoch')
#     axs[i,1].legend()

# plt.tight_layout()
# plt.show()

## Ensembling Submission (Un-official)
I've officially submitted the model inference and results based on single fold (fold-0) results which is only 80% of the data due to shortage of time and limited GPU availability.
Once, the GPU runtime is renewed for me, I've trained the model of fold-1 and fold-2 then uploaded the models on kaggle hub for inference in later versions of this notebook. This is to ensemble the results of the given 3 folds on which the model has been trained to improve the performance of the model. In the last version (V6) of this notebook you can get to see the training logs for the subsequent folds.


In [11]:
# !python main.py --batch-size 64 --epochs 25 --fold 1 --model "optimized" --trackio_name "Pretrained-SOTA-Model-fold_1" --trackio_group "optimized"

In [12]:
# !python main.py --batch-size 64 --epochs 25 --fold 2 --model "optimized" --trackio_name "Pretrained-SOTA-Model-fold_2" --trackio_group "optimized"

In [13]:
import torch
import numpy as np
import pandas as pd
import os
from pathlib import Path
from main import Config, FaceDataModule, FaceModule, BaselineFaceModule, generate_predictions_with_tta, generate_predictions

CHECKPOINT_DIR = Path("/kaggle/input/age-gender-regression-models/pytorch/optimized-model/1")  # Update if needed
FOLDS = [0, 1, 2]
MODELS_PATH = [
    "/kaggle/input/age-gender-regression-models/pytorch/optimized-model/1/optimized_fold0_eepoch14_sval_final_score0.8549.ckpt",
    "/kaggle/input/age-gender-regression-models/pytorch/optimized-model-fold1/1/optimized_f1_eepoch10_sval_final_score0.8539.ckpt",
    "/kaggle/input/age-gender-regression-models/pytorch/optimized-model-fold2/1/optimized_f2_eepoch23_sval_final_score0.8548.ckpt"
]

config = Config()
config.MODEL_TYPE = "optimized" 
data_module = FaceDataModule(config)
data_module.setup()
test_loader = data_module.test_dataloader()

all_age_preds = []
all_gender_preds = []

for ckpt in MODELS_PATH:
    ckpt_path = ckpt 
    print(f"Loading {ckpt_path} ...")
    model = FaceModule.load_from_checkpoint(str(ckpt_path), config=config)
    model.eval()
    fold_df = generate_predictions_with_tta(model.model, test_loader, config=config)
    all_age_preds.append(fold_df['age'].values)
    all_gender_preds.append(fold_df['gender'].values)

# Ensemble: Mean for age, majority vote for gender
ages = np.stack(all_age_preds) 
genders = np.stack(all_gender_preds) 
ids = fold_df['id'].values

final_age = np.clip(np.round(ages.mean(axis=0)), 1, config.NUM_AGE_CLASSES).astype(int)
def majority(arr): 
    vals, counts = np.unique(arr, return_counts=True)
    return vals[np.argmax(counts)]
final_gender = np.apply_along_axis(majority, 0, genders)

ensemble_df = pd.DataFrame({
    "id": ids,
    "age": final_age,
    "gender": final_gender,
})
ensemble_df.to_csv(str(config.OUTPUT_DIR / "submission.csv"), index=False)
print("Ensembled predictions saved to submission.csv")

17:03:49 | INFO | Data | Train: 27766 | Val: 6942 | Test: 8677


Loading /kaggle/input/age-gender-regression-models/pytorch/optimized-model/1/optimized_fold0_eepoch14_sval_final_score0.8549.ckpt ...


17:03:58 | INFO | Loading pretrained weights from Hugging Face hub (timm/convnextv2_base.fcmae_ft_in22k_in1k)


model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

17:04:00 | INFO | [timm/convnextv2_base.fcmae_ft_in22k_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
17:04:01 | INFO | ✓ Gradient checkpointing enabled
17:04:01 | INFO | Running inference with TTA=True
17:06:19 | INFO | Predictions | Age mean: 36.9 ± 15.60
17:06:19 | INFO | Gender | M: 6551 | F: 2126


Loading /kaggle/input/age-gender-regression-models/pytorch/optimized-model-fold1/1/optimized_f1_eepoch10_sval_final_score0.8539.ckpt ...


17:06:28 | INFO | Loading pretrained weights from Hugging Face hub (timm/convnextv2_base.fcmae_ft_in22k_in1k)
17:06:28 | INFO | [timm/convnextv2_base.fcmae_ft_in22k_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
17:06:28 | INFO | ✓ Gradient checkpointing enabled
17:06:28 | INFO | Running inference with TTA=True
17:08:45 | INFO | Predictions | Age mean: 37.2 ± 15.93
17:08:45 | INFO | Gender | M: 6479 | F: 2198


Loading /kaggle/input/age-gender-regression-models/pytorch/optimized-model-fold2/1/optimized_f2_eepoch23_sval_final_score0.8548.ckpt ...


17:08:53 | INFO | Loading pretrained weights from Hugging Face hub (timm/convnextv2_base.fcmae_ft_in22k_in1k)
17:08:53 | INFO | [timm/convnextv2_base.fcmae_ft_in22k_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
17:08:53 | INFO | ✓ Gradient checkpointing enabled
17:08:54 | INFO | Running inference with TTA=True
17:11:10 | INFO | Predictions | Age mean: 36.8 ± 15.27
17:11:10 | INFO | Gender | M: 6563 | F: 2114


Ensembled predictions saved to submission.csv
